# MLP Genetic Algorithm Hyperparameter Optimisation

This notebook uses the same frozen train/validation data, search space, objective, class weights, and 40-unique-evaluation budget as Random Search. Duplicate configurations reuse cached fitness and do not consume another unique model evaluation.


## 1. Package Setup


In [ ]:
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared"
SHARED_MANIFESTS_DIR = SHARED_OUTPUT_DIR / "manifests"
SHARED_CLASS_WEIGHT_PATH = SHARED_OUTPUT_DIR / "class_weights.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
HPO_DIR = OUTPUT_DIR / "hpo"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, HPO_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run Model Variants/Analysis/"
            "MLAAD_Scan_MFCC_Analysis_cache.ipynb, then 00_MLP_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


required_cache_files = [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]
for required in required_cache_files:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")

# Keep these names for older checklist cells, but they now point to cache metadata copied
# from the single canonical shared manifest rather than a model-local split.
train_manifest = train_metadata.copy()
validation_manifest = validation_metadata.copy()

with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
cnn_feature_config = feature_config

CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training arrays, labels and metadata do not have the same row count.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation arrays, labels and metadata do not have the same row count.")

print("Loaded MLP-ready cache:", CACHE_DIR)
print("Input representation:", "80-D MFCC mean/std vector")
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 2. Load Shared MLP-Ready Cache


In [ ]:
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "shared"
SHARED_MANIFESTS_DIR = SHARED_OUTPUT_DIR / "manifests"
SHARED_CLASS_WEIGHT_PATH = SHARED_OUTPUT_DIR / "class_weights.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
HPO_DIR = OUTPUT_DIR / "hpo"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, HPO_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run Model Variants/Analysis/MLAAD_Scan_MFCC_Analysis_cache.ipynb, "
            "then 00_MLP_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")

# Compatibility aliases for older checklist cells. These are cache metadata copied from the
# single canonical shared manifest, not a model-local split.
train_manifest = train_metadata.copy()
validation_manifest = validation_metadata.copy()

with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
cnn_feature_config = feature_config

CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training arrays, labels and metadata do not have the same row count.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation arrays, labels and metadata do not have the same row count.")

print("Loaded MLP-ready cache:", CACHE_DIR)
print("Input representation:", "80-D MFCC mean/std vector")
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 3. Shared MLP Search Space


In [ ]:
SEARCH_SPACE = {
    "hidden_units": [
        [64],
        [128],
        [64, 32],
        [128, 64],
        [256, 128],
        [256, 128, 64],
    ],
    "activation": ["relu", "tanh"],
    "dropout": [0.0, 0.1, 0.2, 0.3, 0.4],
    "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "batch_size": [32, 64, 128, 256],
}
SEARCH_KEYS = ["hidden_units", "activation", "dropout", "learning_rate", "batch_size"]
EXPECTED_SEARCH_SPACE_SIZE = 1200


def normalise_config(config):
    return {
        "hidden_units": [int(value) for value in config["hidden_units"]],
        "activation": str(config["activation"]),
        "dropout": float(config["dropout"]),
        "learning_rate": float(config["learning_rate"]),
        "batch_size": int(config["batch_size"]),
    }


def enumerate_search_space():
    configs = []
    for values in product(*(SEARCH_SPACE[key] for key in SEARCH_KEYS)):
        configs.append(normalise_config(dict(zip(SEARCH_KEYS, values))))
    return configs


ALL_CONFIGURATIONS = enumerate_search_space()
if len(ALL_CONFIGURATIONS) != EXPECTED_SEARCH_SPACE_SIZE:
    raise RuntimeError(f"Expected 1200 configurations, found {len(ALL_CONFIGURATIONS)}")


def config_json(config):
    return json.dumps(normalise_config(config), sort_keys=True)


def config_key(config):
    return hashlib.sha256(config_json(config).encode("utf-8")).hexdigest()


def seed_from_config(config, base_seed=RANDOM_STATE):
    digest = hashlib.sha256(f"{base_seed}:{config_json(config)}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)


def set_global_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def build_mlp_model(config, input_dim):
    config = normalise_config(config)
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    for units in config["hidden_units"]:
        model.add(Dense(units, activation=config["activation"]))
        model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=config["learning_rate"])
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model


def flatten_config(config):
    config = normalise_config(config)
    return {
        "hidden_units": json.dumps(config["hidden_units"]),
        "activation": config["activation"],
        "dropout": config["dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "config_json": config_json(config),
        "config_key": config_key(config),
    }


def train_and_evaluate_config(config, run_seed, run_name, verbose=0):
    config = normalise_config(config)
    tf.keras.backend.clear_session()
    set_global_seed(run_seed)
    model = build_mlp_model(config, X_train.shape[1])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        )
    ]
    start_time = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=verbose,
    )
    runtime_seconds = time.perf_counter() - start_time

    validation_probability = model.predict(X_validation, batch_size=config["batch_size"], verbose=0).ravel()
    validation_pred = (validation_probability >= THRESHOLD).astype(int)
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    best_val_loss = float(np.min(history.history["val_loss"]))

    return {
        "run_name": run_name,
        "seed": int(run_seed),
        "validation_macro_f1": f1_score(y_validation, validation_pred, average="macro", zero_division=0),
        "validation_binary_f1_synthetic": f1_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_accuracy": accuracy_score(y_validation, validation_pred),
        "validation_precision_synthetic": precision_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "validation_recall_synthetic": recall_score(y_validation, validation_pred, pos_label=1, zero_division=0),
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "epochs_trained": int(len(history.history["loss"])),
        "early_stopped": bool(len(history.history["loss"]) < MAX_EPOCHS),
        "runtime_seconds": float(runtime_seconds),
        "objective": "validation_macro_f1",
        "threshold": THRESHOLD,
        **flatten_config(config),
    }


print("Shared HPO search-space size:", len(ALL_CONFIGURATIONS))


## 4. Genetic Algorithm


In [ ]:
RUN_GENETIC_ALGORITHM = False
VERBOSE_TRAINING = 2
GA_MAX_UNIQUE_EVALUATIONS = 40
GA_POPULATION_SIZE = 10
GA_TOURNAMENT_SIZE = 3
GA_ELITE_COUNT = 2
GA_MUTATION_RATE = 0.20
GA_MAX_GENERATIONS = 200

genetic_algorithm_trials_path = HPO_DIR / "genetic_algorithm_trials.csv"
genetic_algorithm_best_path = HPO_DIR / "genetic_algorithm_best.json"


def random_config(rng):
    return normalise_config({key: SEARCH_SPACE[key][int(rng.integers(len(SEARCH_SPACE[key])))] for key in SEARCH_KEYS})


def mutate_config(config, rng):
    child = normalise_config(config)
    for key in SEARCH_KEYS:
        if rng.random() < GA_MUTATION_RATE:
            child[key] = SEARCH_SPACE[key][int(rng.integers(len(SEARCH_SPACE[key])))]
    return normalise_config(child)


def uniform_crossover(parent_a, parent_b, rng):
    return normalise_config({key: parent_a[key] if rng.random() < 0.5 else parent_b[key] for key in SEARCH_KEYS})


def tournament_select(evaluated_rows, rng):
    tournament_size = min(GA_TOURNAMENT_SIZE, len(evaluated_rows))
    tournament_indices = rng.choice(len(evaluated_rows), size=tournament_size, replace=False)
    tournament = [evaluated_rows[int(index)] for index in tournament_indices]
    return max(tournament, key=lambda row: row["validation_macro_f1"])


def evaluate_ga_config(config, fitness_cache, unique_count, generation, individual_index):
    key = config_key(config)
    if key in fitness_cache:
        cached = dict(fitness_cache[key])
        cached.update(
            {
                "method": "genetic_algorithm",
                "generation": generation,
                "individual": individual_index,
                "evaluation_number": len(fitness_cache),
                "cache_hit": True,
            }
        )
        return cached, unique_count

    unique_count += 1
    row = train_and_evaluate_config(
        config,
        run_seed=seed_from_config(config),
        run_name=f"ga_unique_{unique_count:03d}",
        verbose=VERBOSE_TRAINING,
    )
    row.update(
        {
            "method": "genetic_algorithm",
            "generation": generation,
            "individual": individual_index,
            "evaluation_number": unique_count,
            "unique_evaluation_number": unique_count,
            "cache_hit": False,
        }
    )
    fitness_cache[key] = dict(row)
    return row, unique_count


def run_genetic_algorithm():
    rng = np.random.default_rng(RANDOM_STATE)
    population = [random_config(rng) for _ in range(GA_POPULATION_SIZE)]
    fitness_cache = {}
    trial_rows = []
    unique_count = 0
    generation = 0

    while unique_count < GA_MAX_UNIQUE_EVALUATIONS and generation < GA_MAX_GENERATIONS:
        generation += 1
        generation_rows = []

        for individual_index, config in enumerate(population, start=1):
            row, unique_count = evaluate_ga_config(config, fitness_cache, unique_count, generation, individual_index)
            generation_rows.append(row)
            trial_rows.append(row)
            pd.DataFrame(trial_rows).to_csv(genetic_algorithm_trials_path, index=False)
            if unique_count >= GA_MAX_UNIQUE_EVALUATIONS:
                break

        unique_generation_rows = (
            pd.DataFrame(generation_rows)
            .sort_values("validation_macro_f1", ascending=False)
            .drop_duplicates("config_key")
            .to_dict("records")
        )
        elites = [json.loads(row["config_json"]) for row in unique_generation_rows[:GA_ELITE_COUNT]]
        next_population = elites[:]

        while len(next_population) < GA_POPULATION_SIZE and unique_generation_rows:
            parent_a = tournament_select(unique_generation_rows, rng)
            parent_b = tournament_select(unique_generation_rows, rng)
            child = uniform_crossover(json.loads(parent_a["config_json"]), json.loads(parent_b["config_json"]), rng)
            child = mutate_config(child, rng)
            next_population.append(child)

        while len(next_population) < GA_POPULATION_SIZE:
            next_population.append(random_config(rng))

        population = next_population[:GA_POPULATION_SIZE]

    if unique_count < GA_MAX_UNIQUE_EVALUATIONS:
        raise RuntimeError(f"GA stopped after {unique_count} unique evaluations before reaching 40.")

    results_df = pd.DataFrame(trial_rows)
    best_row = results_df.loc[results_df["cache_hit"] == False].sort_values("validation_macro_f1", ascending=False).iloc[0].to_dict()
    with open(genetic_algorithm_best_path, "w", encoding="utf-8") as f:
        json.dump(best_row, f, indent=2)
    return results_df


if RUN_GENETIC_ALGORITHM:
    genetic_algorithm_results_df = run_genetic_algorithm()
    display(genetic_algorithm_results_df.sort_values("validation_macro_f1", ascending=False).head())
elif genetic_algorithm_trials_path.exists():
    print("RUN_GENETIC_ALGORITHM is False. Existing GA trials are available:")
    genetic_algorithm_results_df = pd.read_csv(genetic_algorithm_trials_path)
    display(genetic_algorithm_results_df.sort_values("validation_macro_f1", ascending=False).head())
else:
    print("RUN_GENETIC_ALGORITHM is False. Set it to True when ready to run 40 unique evaluations.")


## 5. Test Set Deliberately Unused


In [ ]:
checks = {
    "uses_frozen_train_manifest": True,
    "uses_frozen_validation_manifest": True,
    "loads_cached_mfcc_features": True,
    "search_space_size": len(ALL_CONFIGURATIONS),
    "unique_evaluation_budget": GA_MAX_UNIQUE_EVALUATIONS,
    "objective": "validation_macro_f1",
    "test_set_loaded": False,
    "next_notebook": "03_MLP_Optimisation_Comparison.ipynb",
}
display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
